In [2]:
# %%capture
!pip install keras_tuner -q
!pip install tensorflow -q

import numpy as np
import matplotlib.pyplot as plt
import keras_tuner as kt
from sklearn.datasets import load_wine, load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Input
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.optimizers import Adam, SGD

# Função genérica para carregar e processar dados
def carregar_dados(dataset_loader):
    data = dataset_loader()
    X = data.data
    y = data.target

    # One-hot encoding apenas se forem mais de 2 classes
    if len(np.unique(y)) > 2:
        y = to_categorical(y)

    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, data

# Função para construir o modelo com hiperparâmetros variáveis
def construir_modelo(hp, input_shape, output_units):
    model = Sequential()
    model.add(Input(shape=(input_shape,)))

    # Número de camadas ocultas
    for i in range(hp.Int("num_camadas_ocultas", 1, 3)):
        model.add(Dense(
            units=hp.Int(f"units_{i}", min_value=4, max_value=64, step=8),
            activation=hp.Choice("activation", ["relu", "tanh"])
        ))

    model.add(Dense(output_units, activation='softmax' if output_units > 1 else 'sigmoid'))

    # Otimizador
    lr = hp.Choice("learning_rate", [1e-2, 1e-3, 1e-4])
    optimizer_name = hp.Choice("optimizer", ["adam", "sgd"])

    if optimizer_name == "adam":
        optimizer = Adam(learning_rate=lr)
    else:
        momentum = hp.Float("momentum", 0.0, 0.9, step=0.1)
        optimizer = SGD(learning_rate=lr, momentum=momentum)

    model.compile(
        optimizer=optimizer,
        loss='categorical_crossentropy' if output_units > 1 else 'binary_crossentropy',
        metrics=['accuracy']
    )
    return model

# Função para rodar Keras Tuner em um dataset
def rodar_tuning(dataset_loader, nome_dataset):
    print(f"\n📊 Treinando modelo para: {nome_dataset}")
    X_train, X_test, y_train, y_test, dataset = carregar_dados(dataset_loader)
    input_shape = X_train.shape[1]
    output_units = y_train.shape[1] if len(y_train.shape) > 1 else 1

    tuner = kt.RandomSearch(
        lambda hp: construir_modelo(hp, input_shape, output_units),
        objective='val_accuracy',
        max_trials=10,
        executions_per_trial=1,
        directory='tuning',
        project_name=nome_dataset
    )

    tuner.search(X_train, y_train, epochs=0, validation_split=0.2, verbose=0)
    best_model = tuner.get_best_models(1)[0]
    best_hp = tuner.get_best_hyperparameters(1)[0]

    # Avaliação
    y_pred = best_model.predict(X_test)
    if output_units == 1:
        y_pred_classes = (y_pred > 0.5).astype("int")
        y_true = y_test
    else:
        y_pred_classes = np.argmax(y_pred, axis=1)
        y_true = np.argmax(y_test, axis=1)

    print("\n🔍 Relatório de classificação:")
    print(classification_report(y_true, y_pred_classes, target_names=dataset.target_names))

    print("🏗️ Arquitetura e Hiperparâmetros Selecionados:")
    for i in range(best_hp.get('num_camadas_ocultas')):
        print(f" - Camada {i+1}: {best_hp.get(f'units_{i}')} neurônios, ativação: {best_hp.get('activation')}")

    print(f" - Otimizador: {best_hp.get('optimizer')}")
    print(f" - Learning Rate: {best_hp.get('learning_rate')}")
    if best_hp.get('optimizer') == 'sgd':
        print(f" - Momentum: {best_hp.get('momentum')}")

    tuner.results_summary()

# Rodar para os dois datasets
rodar_tuning(load_wine, "Wine")
rodar_tuning(load_breast_cancer, "BreastCancer")



📊 Treinando modelo para: Wine
Reloading Tuner from tuning/Wine/tuner0.json
2/2 ━━━━━━━━━━━━━━━━━━━━ 0s 51ms/step


/usr/local/lib/python3.11/dist-packages/keras/src/saving/saving_lib.py:757: UserWarning: Skipping variable loading for optimizer 'adam', because it has 2 variables whereas the saved optimizer has 14 variables. 
  saveable.load_own_variables(weights_store.get(inner_path))



🔍 Relatório de classificação:
              precision    recall  f1-score   support

     class_0       1.00      1.00      1.00        14
     class_1       1.00      1.00      1.00        14
     class_2       1.00      1.00      1.00         8

    accuracy                           1.00        36
   macro avg       1.00      1.00      1.00        36
weighted avg       1.00      1.00      1.00        36

🏗️ Arquitetura e Hiperparâmetros Selecionados:
 - Camada 1: 52 neurônios, ativação: tanh
 - Camada 2: 4 neurônios, ativação: tanh
 - Otimizador: adam
 - Learning Rate: 0.01
Results summary
Results in tuning/Wine
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 00 summary
Hyperparameters:
num_camadas_ocultas: 2
units_0: 52
activation: tanh
learning_rate: 0.01
optimizer: adam
units_1: 4
Score: 1.0

Trial 04 summary
Hyperparameters:
num_camadas_ocultas: 1
units_0: 4
activation: tanh
learning_rate: 0.01
optimizer: adam
units_1: 60
momentum: 0.60000000000000

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/base_tuner.py", line 274, in _try_run_and_update_trial
    self._run_and_update_trial(trial, *fit_args, **fit_kwargs)
  File "/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/base_tuner.py", line 261, in _run_and_update_trial
    self.oracle.update_trial(
  File "/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/oracle.py", line 108, in wrapped_func
    ret_val = func(*args, **kwargs)
              ^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/oracle.py", line 523, in update_trial
    self._check_objective_found(metrics)
  File "/usr/local/lib/python3.11/dist-packages/keras_tuner/src/engine/oracle.py", line 791, in _check_objective_found
    raise ValueError(
ValueError: Objective value missing in metrics reported to the Oracle, expected: ['val_accuracy'], found: dict_keys([])
Traceback (most recent call last

1/4 ━━━━━━━━━━━━━━━━━━━━ 0s 64ms/step

4/4 ━━━━━━━━━━━━━━━━━━━━ 0s 23ms/step

🔍 Relatório de classificação:
              precision    recall  f1-score   support

   malignant       0.98      0.98      0.98        43
      benign       0.99      0.99      0.99        71

    accuracy                           0.98       114
   macro avg       0.98      0.98      0.98       114
weighted avg       0.98      0.98      0.98       114

🏗️ Arquitetura e Hiperparâmetros Selecionados:
 - Camada 1: 28 neurônios, ativação: relu
 - Otimizador: adam
 - Learning Rate: 0.001
Results summary
Results in tuning/BreastCancer
Showing 10 best trials
Objective(name="val_accuracy", direction="max")

Trial 04 summary
Hyperparameters:
num_camadas_ocultas: 1
units_0: 28
activation: relu
learning_rate: 0.001
optimizer: adam
momentum: 0.1
units_1: 60
Score: 0.9890109896659851

Trial 02 summary
Hyperparameters:
num_camadas_ocultas: 2
units_0: 44
activation: relu
learning_rate: 0.0001
optimizer: adam
momentum: 0.7000000000000001
units_1: 4
Score: 0.967